## Оценка эффективности биотехнологических препаратов в растениеводстве на основе дисперсионного анализа (ANOVA)

### 📌 Контекст и проблематика
В сельскохозяйственной биотехнологии критически важно оценивать реальное влияние регуляторов роста и защитных препаратов на стойкость культур к заболеваниям. Отсутствие строгой статистической проверки часто приводит к ошибочному внедрению неэффективных дозировок или перерасходу биопрепаратов. 

Настоящий проект посвящен оценке защитного эффекта фиторегулятора (гиббереллина) разной концентрации против вирусного заболевания (морщинистой мозаики картофеля).

### 🎯 Цель проекта
Провести комплексное математико-статистическое исследование для проверки гипотезы о влиянии концентрации гиббереллина на уровень пораженности культуры, определить статистическую значимость наблюдаемых эффектов и установить наилучшую (оптимальную) дозировку препарата.

### 💡 Практическая ценность результатов
* **Доказательность:** Подтверждение эффективности препарата на основе строгого статистического аппарата с уровнем доверия 99%.
* **Оптимизация ресурсозатрат:** Выявление минимально необходимой концентрации действующего вещества, обеспечивающей максимальное подавление фитопатогена.
* **Масштабируемость:** Создание готового математического алгоритма и скрипта для автоматизированного анализа аналогичных биотехнологических экспериментов.


In [74]:
import pandas as pd 
import numpy as numpy
import seaborn as sns 
import scipy.stats as stats

1 этап: формирование датафрейма

In [43]:
data = {"стандартный вариант": [70, 72, 69, 72], "0,0001%": [58, 62,60, 55], "0,0005":[40,43,46,50], "0,001%": [15, 11, 12, 14]}

In [44]:
df = pd.DataFrame(data)
df = df.transpose()
df

,0,1,2,3
стандартный вариант,70,72,69,72
"0,0001%",58,62,60,55
"0,0005",40,43,46,50
"0,001%",15,11,12,14


находим сумму повторений, средние по вариантам, а также сумма вариантов

In [45]:
df["sum_V"] = df.sum(axis = 1)
df

,0,1,2,3,sum_V
стандартный вариант,70,72,69,72,283
"0,0001%",58,62,60,55,235
"0,0005",40,43,46,50,179
"0,001%",15,11,12,14,52


In [46]:
df.loc["sum_П"] = df.sum(axis = 0)
df

,0,1,2,3,sum_V
стандартный вариант,70,72,69,72,283
"0,0001%",58,62,60,55,235
"0,0005",40,43,46,50,179
"0,001%",15,11,12,14,52
sum_П,183,188,187,191,749


In [47]:
df["x-mean"] = df.iloc[0:4, 0:4].mean(axis = 1)
df

,0,1,2,3,sum_V,x-mean
стандартный вариант,70,72,69,72,283,70.75
"0,0001%",58,62,60,55,235,58.75
"0,0005",40,43,46,50,179,44.75
"0,001%",15,11,12,14,52,13.00
sum_П,183,188,187,191,749,NaN


In [48]:
df.iloc[4, 5] = df["x-mean"].iloc[0:4].mean()
df

,0,1,2,3,sum_V,x-mean
стандартный вариант,70,72,69,72,283,70.7500
"0,0001%",58,62,60,55,235,58.7500
"0,0005",40,43,46,50,179,44.7500
"0,001%",15,11,12,14,52,13.0000
sum_П,183,188,187,191,749,46.8125


Заметим что количество наблюдений = 4, количество экспериментов для всех вариантов также равны 4. Следовательно общее количество экспериментов = 4 * 4

In [49]:
n = 4 # количество повторений (у всех групп одинаковое)
v = 4 # количество вариантов
N = n * v # общее количество наблюдений 

Также необходимо найти
СКО 
СКV
CКП
СКЕ

для этого найдем квадраты каждого наблюдения а так же квадрат суммы каждого варианта

In [50]:
df.columns

Index([0, 1, 2, 3, 'sum_V', 'x-mean'], dtype='object')

In [57]:
import numpy as np
squared = {
    'I': df[0]**2, 
    'II': df[1]**2,
    'III': df[2]**2,
    'IV': df[3]**2,
    'V2': df["sum_V"]**2
}
squared = pd.DataFrame(squared)
squared.iloc[4, 4] = np.nan
squared



,I,II,III,IV,V2
стандартный вариант,4900,5184,4761,5184,80089.0
"0,0001%",3364,3844,3600,3025,55225.0
"0,0005",1600,1849,2116,2500,32041.0
"0,001%",225,121,144,196,2704.0
sum_П,33489,35344,34969,36481,NaN


In [68]:
sum_x_squared = squared.iloc[:4, :4].sum().sum()
sum_v_squared = squared.iloc[:4, 4].sum()
sum_П_squared = squared.iloc[4, :4].sum()
print(sum_x_squared, sum_v_squared, sum_П_squared, sep="\n")




42613
170059.0
140283.0


Поскольку мы будем проводить множественное сравнение после ANOVA то необходимо также вычислить поправку "С"

In [69]:

c = (df.iloc[4, 4]**2)/N
c

np.float64(35062.5625)

переходим к вычислению 

SStotal

SSbetween

SSbetween_cells

SSerror



In [70]:
SStotal = sum_x_squared-c

SSbetween = sum_v_squared/n - c

SSbetween_cells = sum_П_squared/v - c

SSerror = SStotal - SSbetween - SSbetween_cells


проверим сильно ли различаются группы из за влияния фактора или нет

находим для этого коэф свободы для SSbetween и SSerror

In [71]:
df_between = (v-1)  
df_error = (N-1) - ((n-1)+(v-1))

In [73]:
Sv= SSbetween/df_between
Se = SSerror/df_error



Ffact = Sv/Se

Ffact

np.float64(248.23386537126996)

поставить а = 0.05

In [75]:
a= 0.05

In [76]:
Fcrit = stats.f.ppf(1 - a, df_between, df_error)

In [77]:
Ffact > Fcrit

np.True_

Поскольку наблюдаемое больше критического значения мы можем сделать вывод, что между группами присутствует разница. Однако это не отвечает нам на вопрос является ли разница существенной чтобы мы учитывали влияение фактора. Для ответа на вопрос используем НСР(LSD)

найдем Sd(стандартная ошибка разности  выборочных средних)

In [79]:
Sd = np.sqrt((2 * Se)/n)
Sd

np.float64(2.236844255244925)

найдем т табличную по стьюденту и НСР

In [90]:
Fcrit05 = stats.t.ppf(1-0.05, df_error)

Fcrit01 = stats.t.ppf(1-0.01, df_error)

НСР05 = Sd * Fcrit05
НСР01 = Sd * Fcrit01
print(НСР01, НСР05, sep="\n")

6.31111721412414
4.1003881326272795


также рассчитает разность средних d(между базовым и др вариантами)

In [91]:
abs(df['x-mean'].iloc[1:4]- df['x-mean'].iloc[0])

0,0001%    12.00
0,0005     26.00
0,001%     57.75
Name: x-mean, dtype: float64

испытанные дозы препарата гиббереллина снижают пораженность картофеля мозаикой на уровне значимости 0,01. Наилучшим является вариант 4 (концентрация 0,001%), снизивший пораженность с 70,75% до 13,00% (d=57,75%≫НСР01 )